In [3]:
%%capture
!pip install facenet-pytorch

In [6]:
import os
import types
import PIL

# 1. Patch PIL._util manually
if not hasattr(PIL, '_util'):
    PIL._util = types.SimpleNamespace()

# 2. Recreate the removed functions
PIL._util.is_directory = os.path.isdir
PIL._util.is_path = lambda f: isinstance(f, (str, bytes)) or hasattr(f, "__fspath__")

In [7]:
!pip install "numpy<2.0"

In [8]:
%%capture
!pip install --upgrade numpy scipy scikit-learn torchvision

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report # Added for precision, recall, f1-score
from PIL import Image
import numpy as np

from facenet_pytorch import InceptionResnetV1
from torchvision import transforms

In [2]:
class LFWCustomDataset(Dataset):
    """
    Custom Dataset xử lý chuẩn xác ảnh LFW cho FaceNet.
    Xử lý: Grayscale (62x47) -> Normalize uint8 -> Image (RGB) -> Resize 160x160 -> Tensor
    """
    def __init__(self, images, labels, transform=None):
        # images từ fetch_lfw_people là mảng numpy float32, khoảng giá trị [0, 1] hoặc [0, 255]
        # Ta cần đảm bảo đưa về dải 0-255 uint8 để PIL xử lý đúng
        if images.max() <= 1.0:
            images = images * 255.0
        self.images = images.astype(np.uint8)
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_np = self.images[idx]
        # Chuyển đổi numpy array thành PIL Image, sau đó convert sang RGB (3 channels)
        img_pil = Image.fromarray(img_np).convert('RGB')

        if self.transform:
            img_tensor = self.transform(img_pil)
        else:
            img_tensor = transforms.ToTensor()(img_pil)

        return img_tensor, torch.tensor(self.labels[idx], dtype=torch.long)

In [3]:
data_transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.RandomHorizontalFlip(), # Data augmentation nhẹ
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [4]:
class FaceNet128d_V1(nn.Module):
    def __init__(self, num_classes):
        super(FaceNet128d_V1, self).__init__()
        self.backbone = InceptionResnetV1(pretrained='vggface2', classify=False)

        # Linear layer nén 512d về 128d embedding theo yêu cầu đề bài
        self.embedding_layer = nn.Linear(512, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

        # Phân loại cho bài toán hiện tại (LFW classes)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        features = self.backbone(x)          # (Batch, 512)
        embed = self.embedding_layer(features) # (Batch, 128)
        embed = self.relu(embed)
        embed = self.dropout(embed)
        logits = self.classifier(embed)        # (Batch, num_classes)
        return logits

In [5]:
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1   = nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2   = nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        return self.sigmoid(avg_out + max_out)

In [6]:
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv1(x_cat))

In [7]:
class CBAM(nn.Module):
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x

In [8]:
class FaceNet128d_Attention_V2(nn.Module):
    def __init__(self, num_classes):
        super(FaceNet128d_Attention_V2, self).__init__()
        base = InceptionResnetV1(pretrained='vggface2', classify=False)

        # Bóc tách các layer của InceptionResnetV1 để chèn Attention đúng vị trí
        # InceptionResnetV1 của facenet_pytorch kết thúc phần trích xuất không gian tại self.block8
        self.features_up_to_block8 = nn.Sequential(
            base.conv2d_1a, base.conv2d_2a, base.conv2d_2b, base.maxpool_3a,
            base.conv2d_3b, base.conv2d_4a, base.conv2d_4b, base.repeat_1,
            base.mixed_6a, base.repeat_2, base.mixed_7a, base.repeat_3, base.block8
        )

        # Feature map tại block8 của kiến trúc này có 1792 channels
        self.attention = CBAM(in_planes=1792)

        # Các layer pooling và flatten tái hiện lại thiết kế nguyên bản
        self.avgpool_1a = base.avgpool_1a # AdaptiveAvgPool2d(output_size=1)
        self.dropout_base = base.dropout

        # Project về 128d
        self.embedding_layer = nn.Linear(1792, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        # 1. Trích xuất đặc trưng không gian
        x = self.features_up_to_block8(x)     # (Batch, 1792, H', W')

        # 2. Áp dụng CBAM Attention NGAY TRƯỚC Global Pooling
        x = self.attention(x)                 # Tinh chỉnh đặc trưng không gian & kênh

        # 3. Pooling & Flatten
        x = self.avgpool_1a(x)
        x = self.dropout_base(x)
        x = torch.flatten(x, 1)               # (Batch, 1792)

        # 4. Ép về 128d embedding và Phân loại
        embed = self.embedding_layer(x)
        embed = self.relu(embed)
        embed = self.dropout(embed)
        logits = self.classifier(embed)
        return logits

In [9]:
def train_and_eval(model, train_loader, val_loader, criterion, optimizer, device, epochs=3):
    model = model.to(device)

    for epoch in range(epochs):
        # train
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_acc = 100 * train_correct / train_total
        train_loss = train_loss / train_total

        # eval
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        all_preds = [] # For precision, recall, f1-score
        all_labels = [] # For precision, recall, f1-score
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy()) # For precision, recall, f1-score
                all_labels.extend(labels.cpu().numpy()) # For precision, recall, f1-score

        val_acc = 100 * val_correct / val_total
        val_loss = val_loss / val_total

        print(f"Epoch [{epoch+1}/{epochs}] | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

        # Print classification report for precision, recall, f1-score
        print("\nValidation Metrics (Precision, Recall, F1-score):")
        print(classification_report(all_labels, all_preds, zero_division=0))

In [10]:
print("Đang tải dataset LFW...")
# Lấy những người có ít nhất 30 ảnh để đảm bảo bài toán hội tụ tốt trong ví dụ
lfw = fetch_lfw_people(min_faces_per_person=30, color=False)
X = lfw.images
y = lfw.target
num_classes = len(lfw.target_names)
print(f"Số lượng ảnh: {X.shape[0]}, Số class (người): {num_classes}")

Đang tải dataset LFW...
Số lượng ảnh: 2370, Số class (người): 34


In [11]:


import numpy as np
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Khởi tạo Datasets và DataLoaders
train_dataset = LFWCustomDataset(X_train, y_train, transform=data_transform)
val_dataset = LFWCustomDataset(X_val, y_val, transform=data_transform)

batch_size = 16 # Batch size nhỏ vì ảnh 160x160 trên model lớn tốn khá nhiều VRAM
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)



In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sử dụng device: {device}\n")

criterion = nn.CrossEntropyLoss()

Sử dụng device: cuda



In [13]:
print("="*50)
print("TRAIN PHIÊN BẢN 1: FaceNet128d Tiêu Chuẩn")
print("="*50)
model_v1 = FaceNet128d_V1(num_classes=num_classes)

# Kỹ thuật Fine-tuning: Set learning rate nhỏ cho backbone (đã pre-train), lr lớn hơn cho head
optimizer_v1 = optim.Adam([
    {'params': model_v1.backbone.parameters(), 'lr': 1e-4},
    {'params': model_v1.embedding_layer.parameters(), 'lr': 1e-3},
    {'params': model_v1.classifier.parameters(), 'lr': 1e-3}
])
train_and_eval(model_v1, train_loader, val_loader, criterion, optimizer_v1, device, epochs=4)

TRAIN PHIÊN BẢN 1: FaceNet128d Tiêu Chuẩn


  0%|          | 0.00/107M [00:00<?, ?B/s]

Epoch [1/4] | Train Loss: 2.4548, Train Acc: 44.57% | Val Loss: 1.4571, Val Acc: 59.28%

Validation Metrics (Precision, Recall, F1-score):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         5
           1       0.00      0.00      0.00         8
           2       0.00      0.00      0.00         7
           3       0.61      0.94      0.74        18
           4       0.00      0.00      0.00         9
           5       0.66      1.00      0.80        41
           6       0.00      0.00      0.00         5
           7       0.85      1.00      0.92        29
           8       1.00      1.00      1.00       113
           9       0.63      1.00      0.77        22
          10       0.00      0.00      0.00         8
          11       0.00      0.00      0.00         6
          12       0.00      0.00      0.00         6
          13       0.18      1.00      0.30        12
          14       1.00      0.08      0.15       

In [14]:
print("\n" + "="*50)
print("TRAIN PHIÊN BẢN 2: FaceNet128d + CBAM Attention")
print("="*50)
model_v2 = FaceNet128d_Attention_V2(num_classes=num_classes)

optimizer_v2 = optim.Adam([
    {'params': model_v2.features_up_to_block8.parameters(), 'lr': 1e-4},
    {'params': model_v2.attention.parameters(), 'lr': 1e-3},
    {'params': model_v2.embedding_layer.parameters(), 'lr': 1e-3},
    {'params': model_v2.classifier.parameters(), 'lr': 1e-3}
])
train_and_eval(model_v2, train_loader, val_loader, criterion, optimizer_v2, device, epochs=4)


TRAIN PHIÊN BẢN 2: FaceNet128d + CBAM Attention
Epoch [1/4] | Train Loss: 1.3975, Train Acc: 65.14% | Val Loss: 0.1721, Val Acc: 95.57%

Validation Metrics (Precision, Recall, F1-score):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       1.00      0.75      0.86         8
           2       1.00      1.00      1.00         7
           3       1.00      1.00      1.00        18
           4       1.00      1.00      1.00         9
           5       0.98      1.00      0.99        41
           6       1.00      1.00      1.00         5
           7       1.00      0.97      0.98        29
           8       1.00      1.00      1.00       113
           9       1.00      0.95      0.98        22
          10       1.00      1.00      1.00         8
          11       1.00      1.00      1.00         6
          12       0.67      1.00      0.80         6
          13       0.92      0.92      0.92        12
 

In [15]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Số lượng tham số của FaceNet128d Tiêu Chuẩn (model_v1): {count_parameters(model_v1)}")
print(f"Số lượng tham số của FaceNet128d + CBAM Attention (model_v2): {count_parameters(model_v2)}")

Số lượng tham số của FaceNet128d Tiêu Chuẩn (model_v1): 27980377
Số lượng tham số của FaceNet128d + CBAM Attention (model_v2): 23199492
